# **Data Pipeline**

## **Basic Libraries**

In [ ]:
from huggingface_hub import login
login(token="###")
import pandas as pd
import json

## **Loading Dataset**

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("lmsys/lmsys-chat-1m")
print(ds['train'][:5])

## **Filter the Dataset**

### 1. Filter the Language = English

In [ ]:
# Filter for English prompts
if 'language' in ds['train'].column_names: 
    ds = ds.filter(lambda x: x['language'] == 'English')
    print("Filtered dataset where Language = 'English':")
    print(ds)
else:
    print("No language metadata available.")

### 2. Filter out redacted Prompts

In [ ]:
# Filter for unredacted Prompts
if 'turn' in ds['train'].column_names:
        ds = ds.filter(lambda x: x['redacted'] == False)
        print("Filtered dataset where 'redacted' == 'False':")
        print(ds)
else:
        print("Column 'redacted' not found in the 'train' split.")

### 3. Filter for Turns = 2

In [ ]:
# Filter for Amount of Turns
if 'turn' in ds['train'].column_names:
        ds = ds['train'].filter(lambda x: x['turn'] == 2)
        print("Filtered dataset where 'turn' == 2:")
        print(ds)
else:
        print("Column 'turn' not found in the 'train' split.")

### 4. Filter for Harrassment, Violence, etc. = False

In [ ]:
# Filter out entries where any category in 'openai_moderation' is marked as true
if 'openai_moderation' in ds.column_names:
    ds = ds.filter(
        lambda x: not any(
            category
            for moderation_entry in x['openai_moderation']  
            for category in moderation_entry['categories'].values()
        )
    )
    print("Filtered dataset where there is no offensive/unsettling speech:")
    print(ds)
else:
    print("No openai_moderation metadata available.")

### Find out what Prompt Length there should be

In [ ]:
# Extract lengths of user prompts
user_prompt_lengths = [
        len(message['content'].split())
        for x in ds['conversation']
        for message in x
        if message['role'] == 'user'
    ]

# Convert lengths to a DataFrame for analysis
lengths_df = pd.DataFrame(user_prompt_lengths, columns=['prompt_length'])

#### Compute Statistics

In [ ]:
# Compute basic statistics
stats = lengths_df['prompt_length'].describe()
print("Prompt Length Statistics:")
print(stats)
percentiles = lengths_df['prompt_length'].quantile([0.05, 0.10, 0.95])
print("5th, 10th, and 95th Percentiles:")
print(percentiles)
# Compute median instead of mean
median = lengths_df['prompt_length'].median()
print("Median Prompt Length:", median)

#### Plot a Histogram

In [ ]:
import matplotlib.pyplot as plt

# Plot a histogram with a shorter x-axis
plt.figure(figsize=(10, 6))
plt.hist(
    lengths_df['prompt_length'], 
    bins=30, 
    edgecolor='black', 
    range=(0, 500)  # Adjust the range as needed, e.g., from 0 to 50 words
)
plt.title('Distribution of User Prompt Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.show()

#### Plot a Boxplot

In [ ]:
def plot_boxplot(x_range=None):
    # Calculate extra percentiles
    percentiles = lengths_df['prompt_length'].quantile([0.10, 0.95])
    p5 = percentiles.loc[0.10]
    p95 = percentiles.loc[0.95]

    plt.figure(figsize=(10, 6))
    
    # Always use the full dataset
    plt.boxplot(
        lengths_df['prompt_length'],
        vert=False,
        patch_artist=True,
        showmeans=True,
        meanline=True
    )
    
    if x_range:
        plt.xlim(x_range)

    # Add vertical lines for 5% and 95% thresholds
    plt.axvline(x=p5, color='red', linestyle='--', label=f'10th percentile ({p5:.1f})')
    plt.axvline(x=p95, color='blue', linestyle='--', label=f'95th percentile ({p95:.1f})')

    plt.title('Boxplot of User Prompt Lengths')
    plt.xlabel('Number of Words')
    plt.legend()
    plt.show()
    
# Call the function to plot the boxplot
plot_boxplot(x_range=(0, 250))  # Adjust the range as needed

### 5a. Filter out short User Prompts

In [ ]:
# Filter for Prompt length
if 'conversation' in ds.column_names:
    ds = ds.filter(
        lambda x: all(
            len(message['content'].split()) >= 3  # word count is >= 3
            for message in x['conversation']
            if message['role'] == 'user'  # Filter only for user prompts
        )
    )
    print("Filtered dataset with user prompts longer than 2 words:")
    print(ds)
else:
    print("No 'conversation' column available.")

### 5b. Filter out long User Prompts

In [ ]:
# Filter for Prompt length
if 'conversation' in ds.column_names:
    ds = ds.filter(
        lambda x: all(
            len(message['content'].split()) <= 156  # word count is <= 156
            for message in x['conversation']
            if message['role'] == 'user'  # Filter only for user prompts
        )
    )
    print("Filtered dataset with user prompts shorter than 157 words:")
    print(ds)
else:
    print("No 'conversation' column available.")

### 6. Filter out Story and Code Generation

#### Strict Filtering

In [ ]:
# Define strict keywords to filter out
strict_story_keywords = [
    "write a story", "short story", "create a story", "creative story", "generate a story",
    "make up a story", "tell me a story", "write a novel", "fan fiction",
    "fairy tale", "bedtime story", "character development", "tell me about"
]

strict_code_keywords = [
    "write code", "generate code", "python script", "javascript code",
    "html page", "sql query", "regex pattern", "bash script",
    "c++ code", "java code", "python code", "create a function", "write a function", 
    "build a class", "code me", "write me a code", "run this code", "compile", "debug"
]

# Combine all strict keywords
strict_keywords = strict_story_keywords + strict_code_keywords

# Filtering function: check if any strict keyword appears in the user prompt
if 'conversation' in ds.column_names:
    ds = ds.filter(
        lambda x: not any(
            keyword in " ".join(
                [message['content'] for message in x['conversation'] if message['role'] == 'user']
            ).lower()
            for keyword in strict_keywords
        )
    )
    print("Filtered dataset with no story/code generation prompts:")
    print(ds)
else:
    print("No conversation column available.")

#### Soft Filtering

In [ ]:
from transformers import pipeline
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

# 1. Load FLAN-T5 classifier
classifier = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    batch_size=16,
    truncation=True,
)

# 2. Define classification function
def classify_prompt(prompt):
    input_text = f"Classify the following user prompt as one of the following categories: (0) Normal task, (1) Story generation, (2) Code generation/analysis. Respond with only the corresponding number.\n Prompt: {prompt}"
    result = classifier(input_text, max_new_tokens=5)[0]['generated_text']
    return result.strip()

# 3. Define soft keywords
soft_story_keywords = [
    "story", "novel", "fiction", "narrative", "adventure", "plot", "opinion", 
    "thoughts", "generate"
]

soft_code_keywords = [
    "code", "program", "function", "class", "script", "algorithm"   
]

soft_keywords = soft_story_keywords + soft_code_keywords

# Precompile regex pattern
pattern_soft = r'|'.join([re.escape(word) for word in soft_keywords])

# 4. Check if 'conversation' column exists
if 'conversation' in ds.column_names:
    # Lists to collect prompts and classifications
    user_prompts = []
    classifications = []
    soft_indices = []

    print("Filtering user prompts with soft keywords...")

    for idx, example in enumerate(tqdm(ds, desc="Soft filtering")):
        user_text = " ".join(
            [message['content'] for message in example['conversation'] if message['role'] == 'user']
        ).lower()

        # If any soft keyword is found, mark for T5 classification
        if re.search(pattern_soft, user_text):
            user_prompts.append(user_text)
            soft_indices.append(idx)

    print(f"\n{len(user_prompts)} user prompts matched soft keywords and will be classified.")

    # Now classify only soft-flagged prompts
    print("Classifying soft-flagged prompts...")
    for prompt in tqdm(user_prompts, desc="FLAN-T5 classification"):
        classifications.append(classify_prompt(prompt))

    # Create DataFrame with soft-flagged prompts and their classification
    classified_df = pd.DataFrame({
        'prompt': user_prompts,
        'classification': classifications
    })

    # Separate prompts classified as 1 (Story) or 2 (Code)
    creative_or_code_df = classified_df[classified_df['classification'].isin(['1', '2'])]

    # Show the first 10 flagged prompts
    print("\nFirst 10 Story/Creative or Code Generation Prompts:")
    print(creative_or_code_df.head(10))

    # Save flagged prompts
    creative_or_code_df.to_csv('creative_or_code_prompts.csv', index=False)
    print("\n Saved creative/code prompts to 'creative_or_code_prompts.csv'.")

    # Now modify ds:
    # Keep original ds but exclude soft_indices where classified as 1 or 2
    to_remove = [
        soft_indices[i] for i, label in enumerate(classifications) if label in ['1', '2']
    ]

    print(f"\nRemoving {len(to_remove)} creative/code prompts from dataset...")

    keep_indices = list(set(range(len(ds))) - set(to_remove))
    ds = ds.select(keep_indices)

    print("\n Final filtered dataset ready!")
    print(ds)

else:
    print("No conversation column available.")


### Quality Control

In [ ]:
def is_valid_two_turn_conversation(example):
    # Extract all user messages
    user_messages = [
        message for message in example['conversation']
        if message['role'] == 'user' and message.get('content', '').strip()
    ]
    answer_messages = [
        message for message in example['conversation']
        if message['role'] == 'assistant' and message.get('content', '').strip()
    ]
    # Check if there are exactly 2 non-empty user messages
    return len(user_messages) == 2 & len(answer_messages) == 2

# Apply the filter to keep only valid 2-turn conversations
ds = ds.filter(is_valid_two_turn_conversation)

print(f"\n After quality control: {len(ds)} entries with exactly 2 user messages and two answers.")

### Remove Duplicate Conversations

In [ ]:
# Create sets/lists for tracking
seen_pairs = set()
unique_examples = []
duplicate_examples = []

# Loop over all examples in ds
for example in ds:
    # Extract user and assistant messages
    user_messages = [
        msg['content'].strip() for msg in example['conversation']
        if msg['role'] == 'user' and msg.get('content', '').strip()
    ]
    assistant_messages = [
        msg['content'].strip() for msg in example['conversation']
        if msg['role'] == 'assistant' and msg.get('content', '').strip()
    ]

    # Check for two-turn validity
    if len(user_messages) < 2 or len(assistant_messages) < 2:
        continue  # Skip invalid

    # Form pair key
    pair = (user_messages[0], user_messages[1], assistant_messages[0], assistant_messages[1])

    if pair in seen_pairs:
        duplicate_examples.append(example)  # Save for review
    else:
        seen_pairs.add(pair)
        unique_examples.append(example)

# Create new datasets
from datasets import Dataset

ds = Dataset.from_list(unique_examples)
ds_duplicates = Dataset.from_list(duplicate_examples)

print(f"\n Unique two-turn conversations: {len(ds)}")
print(f" Duplicate two-turn conversations: {len(ds_duplicates)}")

In [ ]:
print(ds[:1])

#### Look into a specific ID

In [ ]:
# Replace with the exact user message you're looking for
target_message = "Whats the most recently leaked mathematical model from renaissance technologies?"

# Find the first matching example
for i in range(len(ds)):
    example = ds[i]
    if any(
        msg['role'] == 'user' and msg.get('content', '').strip() == target_message
        for msg in example['conversation']
    ):
        print(example)
        break

### Save the final filtered Dataset

In [ ]:
# Save the filtered dataset
ds.save_to_disk("dataset_lmsys_filtered_v1")

### Optional: Load the saved Dataset

In [ ]:
from datasets import load_from_disk

# Reload the saved dataset
ds = load_from_disk("dataset_lmsys_filtered_v1")
print(ds)

## **Embedding**

### Creating Dataframe

In [ ]:
from datasets import load_from_disk

# Reload the saved dataset
reloaded_ds = load_from_disk("dataset_lmsys_filtered_v1")
print(reloaded_ds)

#### Extract User prompts and Model Answers

In [ ]:
import pandas as pd

def extract_two_turn_conversations(dataset):
    """
    Extract two-turn user prompts and assistant answers from the dataset.

    Args:
        dataset (list): List of conversation datasets

    Returns:
        pandas.DataFrame: DataFrame with 'prompt1', 'answer1', 'prompt2', 'answer2' columns
    """
    prompt1_list = []
    prompt2_list = []
    answer1_list = []
    answer2_list = []

    for conversation in dataset:
        # Extract user and assistant messages
        user_messages = [
            message['content'] for message in conversation['conversation'] if message['role'] == 'user'
        ]
        assistant_messages = [
            message['content'] for message in conversation['conversation'] if message['role'] == 'assistant'
        ]

        # Ensure there are at least two turns
        if len(user_messages) >= 2 and len(assistant_messages) >= 2:
            prompt1_list.append(user_messages[0])  # Extract first user message
            answer1_list.append(assistant_messages[0])  # Extract first assistant response
            prompt2_list.append(user_messages[1])  # Extract second user message
            answer2_list.append(assistant_messages[1])  # Extract second assistant response

    # Create DataFrame
    df = pd.DataFrame({
        'id': range(1, len(prompt1_list)+1),  # Add an ID column
        'prompt1': prompt1_list,
        'answer1': answer1_list,
        'prompt2': prompt2_list,
        'answer2': answer2_list
    })

    return df

In [ ]:
df = extract_two_turn_conversations(reloaded_ds)
print(df)

In [ ]:
print(len(df))

#### Save the DF

In [ ]:
# Save the DataFrame
df.to_csv("DF_v1.csv", index=False)

#### Optional: Load the DF

In [ ]:
import pandas as pd

# Reload to verify
df = pd.read_csv("DF_v1.csv")
print(df.head())

### Sentence Embedding

In [ ]:
!pip install sentence-transformers

In [ ]:
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer('all-mpnet-base-v2')

# Generate embeddings for prompt1 and prompt2 with tqdm
prompt1_embeddings = [
    model.encode(prompt, convert_to_tensor=False) for prompt in tqdm(df['prompt1'], desc="Embedding Prompt1")
]

prompt2_embeddings = [
    model.encode(prompt, convert_to_tensor=False) for prompt in tqdm(df['prompt2'], desc="Embedding Prompt2")
]

# Add embeddings back to the DataFrame
df['prompt1_embeddings'] = prompt1_embeddings
df['prompt2_embeddings'] = prompt2_embeddings

### Save the DF

In [ ]:
# Save the DataFrame
df.to_csv("DF_embedded_v2.csv", index=False)

### Optional: Load the DF

In [ ]:
import pandas as pd

# Reload to verify
df = pd.read_csv("DF_embedded_v2.csv")
print(df.head())

In [ ]:
print(len(df))

## **Similarity Comparison**

### Clean the Embeddings

In [ ]:
import re
import numpy as np

def clean_and_parse_embedding(embedding):
    if isinstance(embedding, np.ndarray):
        # If it's already a numpy array, return it as is
        return embedding
    elif isinstance(embedding, str):
        try:
            # Step 1: Remove extra spaces and ensure proper formatting
            cleaned_str = re.sub(r'\s+', ', ', embedding.strip())  # Replace spaces with commas
            cleaned_str = cleaned_str.replace('[,', '[').replace(', ]', ']')  # Remove leading/trailing commas

            # Step 2: Parse the string into a Python list
            parsed_embedding = eval(cleaned_str)  # Convert the cleaned string to a list
            return np.array(parsed_embedding)  # Convert to a numpy array if needed
        except Exception as e:
            print(f"Error parsing embedding: {embedding}. Error: {e}")
            return None
    else:
        print(f"Unsupported data type: {type(embedding)}")
        return None

# Apply to DataFrame columns
df['prompt1_embeddings'] = df['prompt1_embeddings'].apply(clean_and_parse_embedding)
df['prompt2_embeddings'] = df['prompt2_embeddings'].apply(clean_and_parse_embedding)

### Calculate the Similarity NICHT MACHEN

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def calculate_similarity(row):
    prompt1_embedding = np.array(row['prompt1_embeddings'])
    prompt2_embedding = np.array(row['prompt2_embeddings'])
    return cosine_similarity([prompt1_embedding], [prompt2_embedding])[0][0]

df['prompt_similarity'] = df.apply(calculate_similarity, axis=1)

### Drop Rows where Similarity is too high (>0.9) NICHT MACHEN

In [ ]:
# Drop rows with similarity > 0.9
df = df[df['prompt_similarity'] <= 0.9]

print(f"Remaining rows in the DataFrame: {len(df)}")

### Remove Duplicate Rows (where Prompt 1 and 2 have been seen before)

In [ ]:
# Create an empty set to track seen pairs of (prompt1, prompt2)
seen_pairs = set()

# Function to check if a row is a duplicate based on both 'prompt1' and 'prompt2'
def is_unique_pair(row):
    pair = (row['prompt1'], row['prompt2'], row['answer1'], row['answer2'])
    if pair in seen_pairs:
        return False  # Duplicate found
    seen_pairs.add(pair)  # Add the pair to the seen set
    return True

# Apply the function to filter out duplicate pairs of 'prompt1' and 'prompt2'
df= df[df.apply(is_unique_pair, axis=1)].reset_index(drop=True)

print(f"Filtered dataframe contains {len(df)} rows (removed duplicate pairs).")

### Save the DF

In [ ]:
# Save the DataFrame
df.to_csv("2T_Sim_embedded_DF_final.csv", index=False)

### Optional: Load the DF

In [ ]:
import pandas as pd

# Reload to verify
df = pd.read_csv("2T_Sim_embedded_DF_final.csv")
print(df.head())

In [ ]:
print(len(df["prompt1_embeddings"][0]))

## **PCA**

### Use PCA on Prompt 1 & 2 Embedding

In [ ]:
import pandas as pd
import numpy as np
import ast

# Example DataFrame with the issue
# Replace `your_dataframe.csv` with your actual data source
# df = pd.read_csv("your_dataframe.csv")

def clean_embedding_string(embedding_str):
    # Remove newlines and backslashes
    cleaned_str = embedding_str.replace("\\", "").replace("\n", "").strip()
    
    # Replace multiple spaces with a single space to avoid accidental issues
    cleaned_str = " ".join(cleaned_str.split())
    
    # Replace spaces between numbers with commas
    cleaned_str = cleaned_str.replace(" ", ",")
    
    # Remove any accidental consecutive commas
    cleaned_str = cleaned_str.replace(",,", ",")
    
    # Remove stray commas right after an opening bracket
    cleaned_str = cleaned_str.replace("[,", "[")
    
    # Remove stray commas right before a closing bracket
    cleaned_str = cleaned_str.replace(",]", "]")
    
    # Convert the cleaned string into a list
    return ast.literal_eval(cleaned_str)

# Apply the cleaning function
df["prompt1_embeddings"] = df["prompt1_embeddings"].apply(clean_embedding_string)
df["prompt2_embeddings"] = df["prompt2_embeddings"].apply(clean_embedding_string)

# Check the first few rows
print(df["prompt1_embeddings"].head())

In [ ]:
print(df["prompt1_embeddings"].iloc[0])

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# Stack embeddings into arrays for PCA
prompt1_embeddings = np.array(df['prompt1_embeddings'].tolist())
prompt2_embeddings = np.array(df['prompt2_embeddings'].tolist())

# Fit PCA on prompt embeddings
pca = PCA(n_components=50)  # Choose the number of components based on explained variance
reduced_prompt1_embeddings = pca.fit_transform(prompt1_embeddings)
reduced_prompt2_embeddings = pca.fit_transform(prompt2_embeddings)

# Add reduced embeddings back to the DataFrame
df['prompt1_pca'] = list(reduced_prompt1_embeddings)
df['prompt2_pca'] = list(reduced_prompt2_embeddings)

In [ ]:
print(df)

### Save the DF

In [ ]:
# Save the DataFrame
df.to_csv("2T_PCASim_embedded_DF_final.csv", index=False)

### Optional: Load the DF

In [ ]:
import pandas as pd

# Reload to verify
df = pd.read_csv("2T_PCASim_embedded_DF_final.csv")
print(df.head())

## **Clustering**

#### Reduce Dimensionality with UMAP

In [ ]:
!pip install umap-learn

#### For Prompt 1

In [ ]:
import umap.umap_ as umap

umap_model = umap.UMAP(n_neighbors=15, n_components=2, metric='cosine')
umap_embeddings1 = umap_model.fit_transform(prompt1_embeddings)

#### HDBSCAN

In [ ]:
!pip install hdbscan

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=110, min_samples=5, metric='euclidean', prediction_data=True)
cluster_labels = clusterer.fit_predict(umap_embeddings1)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(umap_embeddings1[:, 0], umap_embeddings1[:, 1], c=cluster_labels, cmap='Spectral', s=10)
plt.title("UMAP + HDBSCAN Clustering")
plt.show()

In [ ]:
import numpy as np
labels = clusterer.labels_
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_ratio = np.sum(labels == -1) / len(labels)
print(f"Clusters: {n_clusters}, Noise: {noise_ratio:.2%}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Create a colormap: use 'Spectral' for clusters, black for noise
unique_labels = np.unique(cluster_labels)
n_clusters = len(unique_labels[unique_labels != -1])

# Create a color map with n_clusters colors + 1 black for noise
base_cmap = plt.get_cmap('Spectral', n_clusters)
colors = [base_cmap(i) for i in range(n_clusters)]
colors.insert(0, (0, 0, 0, 1))  # Add black at index 0 for noise

cmap = ListedColormap(colors)

# Remap cluster labels: -1 (noise) → 0, cluster 0 → 1, cluster 1 → 2, etc.
label_map = {old: new for new, old in enumerate(sorted(unique_labels))}
mapped_labels = np.array([label_map[label] for label in cluster_labels])

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(umap_embeddings1[:, 0], umap_embeddings1[:, 1], c=mapped_labels, cmap=cmap, s=10)
plt.title("UMAP + HDBSCAN Clustering (Noise in Black)")
plt.show()

### DBSCAN for Prompt 1

In [ ]:
import pandas as pd

# Reload to verify
df = pd.read_csv("2T_Sim_embedded_DF_final.csv")
print(df.head())

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
print(type(df["prompt1_embeddings"][0]))

In [ ]:
print(df["prompt1_embeddings"].iloc[0])

#### Adjust Prompt 1

In [ ]:
df["prompt1_embeddings"] = df["prompt1_embeddings"].str.replace(r"(?<!\[)\s+", ",", regex=True)

In [ ]:
print(df["prompt1_embeddings"].iloc[0])

#### String to Lists

In [ ]:
import ast

df["prompt1_embeddings"] = df["prompt1_embeddings"].apply(ast.literal_eval)

#### Lists to Array

In [ ]:
import numpy as np

df["prompt1_embeddings"] = df["prompt1_embeddings"].apply(np.array)

#### Add Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(list(df["prompt1_embeddings"]))

#### Run DBSCAN

In [ ]:
dbscan = DBSCAN(eps=15, min_samples=20, metric='euclidean')
labels = dbscan.fit_predict(X_scaled) 

#### Evaluate the Results

In [ ]:
import numpy as np

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"Estimated number of clusters: {n_clusters}")
print(f"Estimated number of noise points: {n_noise}")

##### Tune Parameters

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

neighbors = NearestNeighbors(n_neighbors=5)  # Use min_samples value
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

distances = np.sort(distances[:, -1])
plt.plot(distances)
plt.title('K-Distance Graph')
plt.xlabel('Points sorted by distance')
plt.ylabel('Distance to 5th Nearest Neighbor')
plt.show()

##### Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score
score = silhouette_score(X_scaled, labels)
print(f"Silhouette Score: {score}")

### K-Means for Prompt 1

#### UMAP

In [ ]:
import pandas as pd
import numpy as np

# Load CSV
df = pd.read_csv('DF_embedded_v2.csv')

# Function to parse space-separated vector strings
def parse_embedding(s):
    try:
        # Remove brackets, split by spaces, convert to float
        s = s.strip('[]')                     # remove square brackets
        return np.array([float(x) for x in s.split()])
    except Exception as e:
        print(f"Failed to parse: {s}\nError: {e}")
        return np.nan  # or handle differently

# Apply to both columns
df['prompt1_embeddings'] = df['prompt1_embeddings'].apply(parse_embedding)
df['prompt2_embeddings'] = df['prompt2_embeddings'].apply(parse_embedding)

# Check: if any failed to parse
print(df['prompt1_embeddings'].isna().sum(), "rows failed to parse.")

In [ ]:
prompt1_embeddings = list(df['prompt1_embeddings'].values)

In [ ]:
import umap.umap_ as umap

umap_model = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)
umap_embeddings1 = umap_model.fit_transform(prompt1_embeddings)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# Range of clusters to test
cluster_range = range(10, 51, 5)
sil_scores = []
inertias = []

for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(umap_embeddings1)  # Use UMAP-reduced embeddings
    inertia = kmeans.inertia_
    score = silhouette_score(umap_embeddings1, labels)

    sil_scores.append(score)
    inertias.append(inertia)

    print(f"Clusters: {k}, Silhouette Score: {score:.4f}, Inertia: {inertia:.2f}")

# Plot Silhouette Score
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(cluster_range, sil_scores, marker='o')
plt.title("Silhouette Score vs. Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.grid(True)

# Plot Inertia (Elbow)
plt.subplot(1, 2, 2)
plt.plot(cluster_range, inertias, marker='o')
plt.title("Elbow Method (Inertia vs. Number of Clusters)")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia (Within-cluster Sum of Squares)")
plt.grid(True)

plt.tight_layout()
plt.show()

#### KMeans with 45 clusters

In [ ]:
from sklearn.cluster import KMeans

n_clusters = 45  # You choose how coarse you want your topics
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(umap_embeddings1)


#### Visualization

In [ ]:
import matplotlib.pyplot as plt

umap_2d = umap.UMAP(n_neighbors=15, n_components=2, metric='cosine').fit_transform(prompt1_embeddings)

plt.scatter(umap_2d[:, 0], umap_2d[:, 1], c=cluster_labels, cmap='Spectral', s=10)
plt.title(f"K-Means Clustering with {n_clusters} Clusters")
plt.show()

#### Add the Clusters to the DF

In [ ]:
df['cluster1'] = cluster_labels
print(df['cluster1'].head())

#### Extract Top Terms for Topic Naming

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Set how many keywords per cluster
TOP_N = 25

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df['prompt1'])
feature_names = vectorizer.get_feature_names_out()

# Build dictionary: cluster_id → top keywords
top_keywords = {}

for cluster_id in sorted(df['cluster1'].unique()):
    cluster_indices = df[df['cluster1'] == cluster_id].index
    cluster_tfidf = tfidf_matrix[cluster_indices].mean(axis=0).A1
    top_indices = cluster_tfidf.argsort()[-TOP_N:][::-1]
    keywords = [feature_names[i] for i in top_indices]
    top_keywords[cluster_id] = keywords

In [ ]:
for cluster_id, keywords in top_keywords.items():
    print(f" Cluster {cluster_id}: {', '.join(keywords)}")

#### Count Entries per Cluster

In [ ]:
cluster_counts = df['cluster1'].value_counts().sort_index()
print(cluster_counts)

### Kmeans for Prompt 2

#### UMAP

In [ ]:
# Check: if any failed to parse
print(df['prompt2_embeddings'].isna().sum(), "rows failed to parse.")

In [ ]:
prompt2_embeddings = list(df['prompt2_embeddings'].values)

In [ ]:
import umap.umap_ as umap

umap_model = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)
umap_embeddings2 = umap_model.fit_transform(prompt2_embeddings)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# Range of clusters to test
cluster_range = range(10, 51, 5)
sil_scores = []
inertias = []

for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(umap_embeddings2)  # Use UMAP-reduced embeddings
    inertia = kmeans.inertia_
    score = silhouette_score(umap_embeddings2, labels)

    sil_scores.append(score)
    inertias.append(inertia)

    print(f"Clusters: {k}, Silhouette Score: {score:.4f}, Inertia: {inertia:.2f}")

# Plot Silhouette Score
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(cluster_range, sil_scores, marker='o')
plt.title("Silhouette Score vs. Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.grid(True)

# Plot Inertia (Elbow)
plt.subplot(1, 2, 2)
plt.plot(cluster_range, inertias, marker='o')
plt.title("Elbow Method (Inertia vs. Number of Clusters)")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia (Within-cluster Sum of Squares)")
plt.grid(True)

plt.tight_layout()
plt.show()

#### Kmeans with 20 Clusters

In [ ]:
from sklearn.cluster import KMeans

n_clusters = 20  # You choose how coarse you want your topics
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
cluster_labels2 = kmeans.fit_predict(umap_embeddings2)

#### Visualization

In [ ]:
import matplotlib.pyplot as plt

umap_2d = umap.UMAP(n_neighbors=15, n_components=2, metric='cosine').fit_transform(prompt2_embeddings)

plt.scatter(umap_2d[:, 0], umap_2d[:, 1], c=cluster_labels2, cmap='Spectral', s=10)
plt.title(f"K-Means Clustering with {n_clusters} Clusters")
plt.show()

#### Add Cluster to DF

In [ ]:
print(len(df))
df['cluster2'] = cluster_labels2

In [ ]:
print(len(df['prompt2']))

#### Extract Top Terms for Topic Naming

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Set how many keywords per cluster
TOP_N = 25

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df['prompt2'])
feature_names = vectorizer.get_feature_names_out()

# Build dictionary: cluster_id → top keywords
top_keywords = {}

for cluster_id in sorted(df['cluster2'].unique()):
    cluster_indices = df[df['cluster2'] == cluster_id].index
    cluster_tfidf = tfidf_matrix[cluster_indices].mean(axis=0).A1
    top_indices = cluster_tfidf.argsort()[-TOP_N:][::-1]
    keywords = [feature_names[i] for i in top_indices]
    top_keywords[cluster_id] = keywords

In [ ]:
for cluster_id, keywords in top_keywords.items():
    print(f" Cluster {cluster_id}: {', '.join(keywords)}")

#### Count Entries per Cluster

In [ ]:
cluster_counts = df['cluster2'].value_counts().sort_index()
print(cluster_counts)

#### Save clustered DF

In [ ]:
print(len(df))

In [ ]:
# Save the DataFrame
df.to_csv("DF_clustered_v2.csv", index=False)

In [ ]:
print(df.head())

### Drop Rows with Story or Code Generation

#### Prompt 1

In [ ]:
df = df[~df['cluster1'].isin([0, 3, 5, 6, 8, 9, 10, 11, 13, 18, 23, 24, 25, 28, 29, 32, 33, 42, 43, 44])]

In [ ]:
cluster_counts = df['cluster1'].value_counts().sort_index()
print(cluster_counts)

In [ ]:
print(len(df))

In [ ]:
import matplotlib.pyplot as plt

cluster_counts.plot(kind='bar', figsize=(12, 5))
plt.title("Number of Entries per Cluster")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Prompts")
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

#### Prompt 2

In [ ]:
df = df[~df['cluster2'].isin([0, 3, 5, 7, 12, 14, 18, 19])]

In [ ]:
cluster_counts = df['cluster2'].value_counts().sort_index()
print(cluster_counts)

In [ ]:
print(len(df))

#### Prompt2

In [ ]:
import matplotlib.pyplot as plt

cluster_counts.plot(kind='bar', figsize=(12, 5))
plt.title("Number of Entries per Cluster")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Prompts")
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

#### Prompt 1

In [ ]:
cluster_counts = df['cluster1'].value_counts().sort_index()
print(cluster_counts)

In [ ]:
import matplotlib.pyplot as plt

cluster_counts.plot(kind='bar', figsize=(12, 5))
plt.title("Number of Entries per Cluster")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Prompts")
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

#### Save filtered and clustered DF

In [ ]:
print(len(df))

In [ ]:
# Save the DataFrame
df.to_csv("DF_clusterfiltered_v2.csv", index=False)

### Create a 200 entries subset

In [ ]:
import pandas as pd

# Reload to verify
df = pd.read_csv("DF_clusterfiltered_v2.csv")
print(df.head())
print(len(df))

In [ ]:
##### Is this neccessary here?
from sklearn.cluster import KMeans

n_clusters = 45  # You choose how coarse you want your topics
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(umap_embeddings1)

In [ ]:
from sklearn.metrics import pairwise_distances_argmin_min
import numpy as np
import pandas as pd

# Total number of entries to sample
SUBSET_SIZE = 400

# Step 1: Proportions by cluster
cluster_counts = df['cluster1'].value_counts(normalize=True).sort_index()
samples_per_cluster = (cluster_counts * SUBSET_SIZE).round().astype(int)

# Adjust to sum exactly to SUBSET_SIZE
diff = SUBSET_SIZE - samples_per_cluster.sum()
if diff != 0:
    samples_per_cluster[samples_per_cluster.idxmax()] += diff

# Step 2: Compute distances to cluster centers
df['distance_to_center'] = np.nan
for cluster_id in samples_per_cluster.index:
    cluster_idx = df[df['cluster1'] == cluster_id].index
    cluster_embeddings = umap_embeddings1[cluster_idx]
    center = kmeans.cluster_centers_[cluster_id].reshape(1, -1)
    _, distances = pairwise_distances_argmin_min(cluster_embeddings, center)
    df.loc[cluster_idx, 'distance_to_center'] = distances

# Step 3: Select closest N entries per cluster
central_samples = []
for cluster_id, n_samples in samples_per_cluster.items():
    cluster_data = df[df['cluster1'] == cluster_id]
    top_n = cluster_data.nsmallest(n_samples, 'distance_to_center')
    central_samples.append(top_n)

# Step 4: Combine and clean
df_central_subset = pd.concat(central_samples).reset_index(drop=True)
df_central_subset = df_central_subset.drop(columns=['distance_to_center'])

print(f"Final subdataset created with {len(df_central_subset)} representative entries.")

In [ ]:
print(df_central_subset.head(10))

#### Kick out the embeddings

In [ ]:
df_central_subset = df_central_subset.drop(columns=['prompt1_embeddings', 'prompt2_embeddings'])

#### Save Subset as CSV

In [ ]:
# For the randomly sampled version
df_central_subset.to_csv("subset_DF_v2.csv", index=False)

# Or for the centrally sampled version
# df_central_subset.to_csv("cluster_central_subset.csv", index=False)

#### Save Subset as JSON File

In [ ]:
import csv
import json

def csv_to_filtered_json(csv_path, json_path):
    filtered_data = []

    with open(csv_path, mode='r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            filtered_entry = {
                "id": row.get("id", ""),
                "prompt1": row.get("prompt1", ""),
                "answer1": row.get("answer1", ""),
                "prompt2": row.get("prompt2", ""),
                "answer2": row.get("answer2", "")
            }
            filtered_data.append(filtered_entry)

    with open(json_path, mode='w', encoding='utf-8') as jsonfile:
        json.dump(filtered_data, jsonfile, indent=4, ensure_ascii=False)

csv_to_filtered_json('subset_DF_v2.csv', 'subset_conversations_v2.json')

In [ ]:
import pandas as pd

# Load the CSV file
csv_file = 'subset_DF_v2.csv'          # Replace with your CSV filename
excel_file = 'subset_v2.xlsx'      # Desired Excel output filename

# Read CSV
df = pd.read_csv(csv_file)

# Save as Excel
df.to_excel(excel_file, index=False, engine='openpyxl')

print(f"Converted '{csv_file}' to '{excel_file}' successfully.")

In [ ]:
import pandas as pd

# File paths (replace with your actual filenames)
csv_file_1 = 'subset_DF_v1.csv'
csv_file_2 = 'subset_DF_v2.csv'

# Read both CSV files
df1 = pd.read_csv(csv_file_1)
df2 = pd.read_csv(csv_file_2)

# Ensure 'id' column exists
if 'id' not in df1.columns or 'id' not in df2.columns:
    raise ValueError("Both CSVs must contain an 'id' column.")

# Convert to sets for fast comparison
ids1 = set(df1['id'])
ids2 = set(df2['id'])

# Compare
common_ids = ids1 & ids2
only_in_1 = ids1 - ids2
only_in_2 = ids2 - ids1

# Print results
print(f"Common IDs ({len(common_ids)}): {sorted(common_ids)}\n")
print(f"IDs only in {csv_file_1} ({len(only_in_1)}): {sorted(only_in_1)}\n")
print(f"IDs only in {csv_file_2} ({len(only_in_2)}): {sorted(only_in_2)}\n")

##### Change Seperator in csv from , to ;

In [ ]:
import csv

# Open the original CSV with ',' separator
with open('subset_DF_v1.csv', 'r', newline='', encoding='utf-8') as infile:
    reader = csv.reader(infile, delimiter=',')
    rows = list(reader)

# Write a new CSV with ';' separator
with open('subset__sep_DF_v1.csv', 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile, delimiter=';')
    writer.writerows(rows)

#### Add Annotators randomly

In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd
import random
from collections import defaultdict

# === PARAMETERS ===
input_file = "subset_DF_v1.csv"  # Replace with your CSV file
output_file = "annotator_DF_v1.xlsx"
num_entries = 200
num_annotators = 8
entries_per_annotator = 50
annotations_per_entry = 2
annotators = [f"A{i+1}" for i in range(num_annotators)]

# === LOAD DATA ===
df = pd.read_csv(input_file)
df_subset = df.head(num_entries).copy()  # use only first 200 entries

# === ASSIGNMENT LOGIC ===
entry_assignments = defaultdict(list)
annotator_counts = {a: 0 for a in annotators}
all_entries = list(df_subset.index)
random.shuffle(all_entries)

for entry in all_entries:
    possible_annotators = [a for a in annotators if annotator_counts[a] < entries_per_annotator]
    random.shuffle(possible_annotators)

    assigned = []
    for annotator in possible_annotators:
        if len(assigned) < annotations_per_entry:
            assigned.append(annotator)
            annotator_counts[annotator] += 1
        if len(assigned) == annotations_per_entry:
            break

    entry_assignments[entry] = assigned

# === STRUCTURE OUTPUT DATA ===
records = []
for entry_idx, assigned_annotators in entry_assignments.items():
    row = df_subset.loc[entry_idx]
    for annotator in assigned_annotators:
        records.append({
            "Entry ID": row["id"],
            "Prompt 1": row.get("prompt1", ""),
            "Answer 1": row.get("answer1", ""),
            "Prompt 2": row.get("prompt2", ""),
            "Answer 2": row.get("answer2", ""),
            "Annotator": annotator,
            "Label": ""  # blank for manual input
        })

annot_df = pd.DataFrame(records)

# === SAVE TO EXCEL ===
annot_df.to_excel(output_file, index=False)
print(f"Assignment saved to: {output_file}")

### K-Means for Prompt 2

#### Clean the PCA Embeddings

In [ ]:
import re

def clean_and_parse_pca_embedding(embedding_str):
    try:
        # Normalize the string and replace spaces with commas
        cleaned_str = re.sub(r'\s+', ',', embedding_str.strip())
        cleaned_str = cleaned_str.replace('[,', '[').replace(',]', ']')
        return eval(cleaned_str)
    except Exception as e:
        print(f"Failed to parse: {embedding_str}")
        return None

# Apply the parsing function again
df['prompt2_pca'] = df['prompt2_pca'].apply(clean_and_parse_pca_embedding)

#### Use K-Means

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# Convert the column to a NumPy array for KMeans
reduced_prompt2_embeddings = np.array(df['prompt2_pca'].tolist())

# Perform KMeans clustering
num_clusters = 20 - len(clusters1_to_remove)  # Define the number of clusters - Number of Clusters that have been removed
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
kmeans.fit(reduced_prompt2_embeddings)

# Add the cluster labels to the DataFrame
df['Cluster2'] = kmeans.labels_

print("KMeans clustering completed and cluster labels added to the DataFrame.")

#### Sample the Entries in the Clusters

In [ ]:
import pandas as pd
import numpy as np

# Define the number of samples to take per cluster
samples_per_cluster = 5
closest_points = 100  # Number of closest points to consider for sampling

# Initialize an empty list to store sampled rows
sampled_rows = []

# Iterate through each cluster
for cluster in df['Cluster2'].unique():
    # Select points in the current cluster
    cluster_points = df[df['Cluster2'] == cluster]

    # Compute the centroid of the cluster (mean of embeddings component-wise)
    cluster_centroid = np.mean(np.stack(cluster_points['prompt2_pca']), axis=0)

    # Calculate the distance of each point to the centroid
    cluster_points['distance_to_centroid'] = cluster_points['prompt2_pca'].apply(
        lambda x: np.linalg.norm(np.array(x) - cluster_centroid)
    )

    # Sort points by distance to the centroid
    cluster_points = cluster_points.sort_values(by='distance_to_centroid').head(closest_points)

    # Sample 5 points from the closest 100 points
    cluster_samples = cluster_points.sample(
        n=samples_per_cluster, replace=False, random_state=42
    )
    
    # Append sampled rows
    sampled_rows.append(cluster_samples)
    print(f"Cluster {cluster} done!")

# Combine all samples into a single DataFrame
sampled_df = pd.concat(sampled_rows, ignore_index=True)

# Keep only the specified columns
columns_to_keep = ['prompt2', 'answer2', 'Cluster2']
sampled_df = sampled_df[columns_to_keep]

# Save to an Excel file
output_path = 'samples_cluster2_t2_final.xlsx'
sampled_df.to_excel(output_path, index=False)

print(f"Sampled conversations saved to {output_path}")

#### After inspecting Clusters: Which should be removed?

In [ ]:
# Define the clusters to remove
clusters2_to_remove = [6, 8]

# Filter the DataFrame to exclude rows with these cluster values
df = df[~df['Cluster2'].isin(clusters2_to_remove)]

print(f"Rows with clusters {clusters2_to_remove} removed. Remaining rows: {len(df)}")

### Save the DF

In [ ]:
# Save the DataFrame
df.to_csv("2T_clustered_PCASim_embedded_DF_final.csv", index=False)

### Optional: Load the DF

In [ ]:
import pandas as pd

# Reload to verify
df = pd.read_csv("2T_clustered_PCASim_embedded_DF_final.csv")
print(df.head())

## **Find Topics for remaining Clusters with ChatGPT** 

Prompt:
can you help me identify cluster topics based on 5 samples of each cluster?
you should read all prompts from one cluster first and then decide what a collective cluster topic based on the 5 samples for that cluster is. please do this for all 20 clusters.

### First check for topics of Cluster 1

Cluster 0: Trivia and general knowledge questions.
Cluster 1: Technology and software/hardware-related queries.
Cluster 2: Programming examples and technical tasks.
Cluster 3: Philosophical questions about the meaning of life.
Cluster 4: Business and professional development.
Cluster 5: Random technical and knowledge-based queries.
Cluster 6: Technical troubleshooting, personal advice, and biology-related questions.
Cluster 7: Greeting and self-introduction exchanges.
Cluster 8: Informal and creative prompts.
Cluster 9: Structured tasks and linguistic processing in natural language.
Cluster 10: Scientific and conceptual explanations.
Cluster 11: Python programming tasks.
Cluster 12: Fun, jokes, and misconceptions.
Cluster 13: Academic and analytical queries.
Cluster 14: Mathematical sequences and problem-solving.
Cluster 15: Day-to-day activities and practical advice.
Cluster 16: Artificial intelligence, machine learning, and innovation.
Cluster 17: Friendly greetings and inquiries about well-being.
Cluster 18: Creative writing: poems, lyrics, and surreal compositions.
Cluster 19: Professional communication, grammar corrections, and writing tasks.

### Check Topics for Cluster 2

Cluster 0: Arithmetic and mathematical calculations.
Cluster 1: Geographical and travel-related questions.
Cluster 2: Definitions and explanations of specialized terms or concepts.
Cluster 3: Creative writing requests, such as essays, songs, and introductions.
Cluster 4: Software development, coding, and multimedia tasks.
Cluster 5: Multilingual and international queries.
Cluster 6: Python programming tasks and technical scripts.
Cluster 7: General knowledge, misconceptions, and random questions.
Cluster 8: Technical troubleshooting and system-related problems.
Cluster 9: Database, SQL, and structured data queries.
Cluster 10: Business, strategic foresight, and omni-channel development.
Cluster 11: Ethics, societal issues, and philosophical discussions.
Cluster 12: Educational explanations of advanced topics.
Cluster 13: Investment, hardware, and consumer product queries.
Cluster 14: Lifestyle, mental health, and personal growth.
Cluster 15: Current events and specific information requests.
Cluster 16: Machine learning, AI, and neural network discussions.
Cluster 17: Proofreading, academic writing, and linguistic corrections.
Cluster 18: Medical and health-related questions.
Cluster 19: Informal, personal, or conversational prompts.

### Distribution in Cluster1

In [ ]:
# Count the number of entries in each cluster of 'Cluster1'
cluster1_counts = df['Cluster1'].value_counts().reset_index()

# Rename the columns for clarity
cluster1_counts.columns = ['Cluster', 'Count']

# Calculate the percentage for each cluster
total_entries = df.shape[0]
cluster1_counts['Percentage'] = (cluster1_counts['Count'] / total_entries) * 100

# Sort by cluster number (optional)
cluster1_counts = cluster1_counts.sort_values(by='Cluster').reset_index(drop=True)

# Display the counts and percentages
print(cluster1_counts)

# Save to an Excel file if needed
output_path = 'cluster1_entry_counts_with_percentages.xlsx'
cluster1_counts.to_excel(output_path, index=False)

print(f"Cluster counts with percentages saved to {output_path}")

In [ ]:
import pandas as pd

# Your existing DataFrame with counts and percentages
data = {
    'Cluster': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
    'Count': [1390, 1364, 1267, 139, 1645, 1338, 2472, 308, 1510, 1039, 1954, 448, 1441, 1612, 1267, 1507, 949, 428, 1025, 1319],
    'Percentage': [5.691590, 5.585128, 5.187945, 0.569159, 6.735730, 5.478667, 10.122021, 1.261158, 6.182950, 4.254361, 8.000983, 1.834412, 5.900418, 6.600606, 5.187945, 6.170666, 3.885841, 1.752518, 4.197035, 5.400868]
}

df = pd.DataFrame(data)

# Round Count and Percentage columns
df['Count'] = df['Count'].apply(lambda x: int(round(x)))
df['Percentage'] = df['Percentage'].apply(lambda x: round(x))

# Display the rounded DataFrame
print(df)

### Sample 100 Entries based on Cluster Distribution

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics.pairwise import cosine_similarity

# Function to clean the embeddings
def clean_embedding_string(embedding_str):
    # Remove newlines and backslashes
    cleaned_str = embedding_str.replace("\\", "").replace("\n", "").strip()
    
    # Replace multiple spaces with a single space to avoid accidental issues
    cleaned_str = " ".join(cleaned_str.split())
    
    # Replace spaces between numbers with commas
    cleaned_str = cleaned_str.replace(" ", ",")
    
    # Remove any accidental consecutive commas
    cleaned_str = cleaned_str.replace(",,", ",")
    
    # Remove stray commas right after an opening bracket
    cleaned_str = cleaned_str.replace("[,", "[")
    
    # Remove stray commas right before a closing bracket
    cleaned_str = cleaned_str.replace(",]", "]")
    
    # Convert the cleaned string into a list
    return ast.literal_eval(cleaned_str)

# Rounded percentages provided
clusters = pd.DataFrame({
    'Cluster': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
    'Count': [1390, 1364, 1267, 139, 1645, 1338, 2472, 308, 1510, 1039, 1954, 448, 1441, 1612, 1267, 1507, 949, 428, 1025, 1319],
    'Percentage': [6, 6, 5, 1, 7, 5, 10, 1, 6, 4, 8, 2, 6, 7, 5, 6, 4, 2, 4, 5]
})

# Assuming the 'prompt1_embeddings' and 'Cluster1' columns exist in the original DataFrame
# Replace this with your actual data
original_df = pd.read_csv("2T_clustered_PCASim_embedded_DF_final.csv")

# Apply the cleaning function to the 'prompt1_embeddings' and 'prompt2_embeddings' columns
original_df['prompt1_embeddings'] = original_df['prompt1_embeddings'].apply(clean_embedding_string)
original_df['prompt2_embeddings'] = original_df['prompt2_embeddings'].apply(clean_embedding_string)

# Define total sample size and other parameters
total_samples = 100
sampled_rows = []

# Iterate over each cluster to sample based on the rounded percentages
for idx, row in clusters.iterrows():
    cluster = row['Cluster']
    percentage = row['Percentage']
    
    # Calculate how many samples to take from this cluster
    num_samples = int(np.round(percentage / 100 * total_samples))
    
    # Get rows from the original DataFrame for the current cluster
    cluster_rows = original_df[original_df['Cluster1'] == cluster]
    
    # Convert the embeddings into NumPy arrays
    cluster_embeddings = np.array(cluster_rows['prompt1_embeddings'].tolist())
    
    # Calculate the centroid (mean embedding) for the cluster
    centroid = np.mean(cluster_embeddings, axis=0)
    
    # Calculate the distance of each point to the centroid
    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    
    # Add a new column with the distance to the centroid
    cluster_rows['distance_to_centroid'] = distances
    
    # Sort by distance to centroid (ascending order), and select the closest `num_samples`
    closest_rows = cluster_rows.sort_values(by='distance_to_centroid').head(num_samples)
    
    # Append the selected rows
    sampled_rows.append(closest_rows)

# Combine all sampled rows into a single DataFrame
final_sampled_df = pd.concat(sampled_rows)

# Ensure the final sampled DataFrame has exactly 100 rows
final_sampled_df = final_sampled_df.head(total_samples)

# Keep only the required columns in the final sample
final_sampled_df = final_sampled_df[['prompt1', 'answer1', 'prompt2', 'answer2']]

# Save to Excel
output_path = 'sampled_clusters_100dist_final.xlsx'
final_sampled_df.to_excel(output_path, index=False)

print(f"Sampled data has been saved to {output_path}")

## Label Studio

### Converst XLSX to JSON

In [ ]:
import pandas as pd

# Function to read Excel file and output as JSON
def excel_to_json(input_excel_file, output_json_file):
    # Read the Excel file
    df = pd.read_excel(input_excel_file)

    # Convert the DataFrame to JSON lines format
    df.to_json(output_json_file, orient='records', lines=True)

    print(f"Data has been successfully saved to {output_json_file}")

# Example usage
input_excel_file = 'sampled_clusters_100dist_final.xlsx'  # Replace with your actual Excel file path
output_json_file = 'sampled_clusters_100dist_final_studio.json'  # Replace with the desired output JSON file path

# Convert Excel to JSON
excel_to_json(input_excel_file, output_json_file)

### Convert XLSX to CSV

In [ ]:
import pandas as pd

# Function to read Excel file and output as CSV
def excel_to_csv(input_excel_file, output_csv_file):
    # Read the Excel file
    df = pd.read_excel(input_excel_file)

    # Convert the DataFrame to a CSV file
    df.to_csv(output_csv_file, index=False)

    print(f"Data has been successfully saved to {output_csv_file}")

# Example usage
input_excel_file = 'sampled_clusters_100dist_final.xlsx'  # Replace with your actual Excel file path
output_csv_file = 'sampled_clusters_100dist_final_studio.csv'  # Replace with the desired output CSV file path

# Convert Excel to CSV
excel_to_csv(input_excel_file, output_csv_file)

## NER Extraction

### Extract NER from 100 Samples using BERT

In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForTokenClassification
from transformers import pipeline

# Load the CSV file into a DataFrame (replace with the actual path to your CSV file)
df = pd.read_csv('sampled_clusters_100dist_final_studio.csv')

# Load pre-trained BERT model and tokenizer for NER
model_name = "dslim/bert-base-NER-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForTokenClassification.from_pretrained(model_name)

# Create a NER pipeline
print("Creating Pipeline...")
nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer)

# Function to apply NER to a prompt
def apply_ner(prompt):
    ner_results = nlp_ner(prompt)
    # Extracting entity labels (only the entity words, not the entire result object)
    entities = [entity['word'] for entity in ner_results]
    return ", ".join(entities)  # Join entities by commas

# Apply NER to 'prompt1' and 'prompt2', and store results in new columns 'ner1' and 'ner2'
print("Applying NER to Prompt 1...")
df['ner1'] = df['prompt1'].apply(apply_ner)
print("Applying NER to Prompt 2...")
df['ner2'] = df['prompt2'].apply(apply_ner)

# Save the modified DataFrame to a new CSV file (or overwrite the original file)
df.to_csv('sampled_clusters_100dist with_ner_final_studio.csv', index=False)

# Output a message confirming the update
print("NER applied and saved to 'sampled_clusters_100dist with_ner_final_studio.csv'.")

#### CSV zu EXCEL ändern

In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
csv_file_path = 'sampled_clusters_100dist with_ner_final_studio.csv'  # Replace with your CSV file path
df = pd.read_csv(csv_file_path)

# Save the DataFrame to an Excel file
excel_file_path = 'sampled_clusters_100dist with_ner_final_studio.xlsx'  # Replace with the desired output file path
df.to_excel(excel_file_path, index=False)

print(f"Excel file saved as {excel_file_path}")

### Usage of Flair for NER

In [ ]:
!pip install flair

In [ ]:
import pandas as pd
from flair.data import Sentence
from flair.models import SequenceTagger

# Load the CSV file into a DataFrame (replace with the actual path to your CSV file)
df = pd.read_csv('sampled_clusters_100dist_final_studio.csv')

# Load pre-trained Flair NER model
print("Loading Flair NER model...")
tagger = SequenceTagger.load('ner')

# Function to apply NER to a prompt
def apply_ner(prompt):
    sentence = Sentence(prompt)
    tagger.predict(sentence)
    # Extracting entity words and their labels
    entities = [f"{entity.text} ({entity.get_label('ner').value})" for entity in sentence.get_spans('ner')]
    return ", ".join(entities)  # Join entities by commas

# Apply NER to 'prompt1' and 'prompt2', and store results in new columns 'ner1' and 'ner2'
print("Applying NER to Prompt 1...")
df['ner1'] = df['prompt1'].apply(apply_ner)
print("Applying NER to Prompt 2...")
df['ner2'] = df['prompt2'].apply(apply_ner)

# Save the modified DataFrame to a new CSV file (or overwrite the original file)
df.to_csv('sampled_clusters_100dist_with_ner_flair_final_studio.csv', index=False)

# Output a message confirming the update
print("NER applied and saved to 'sampled_clusters_100dist_with_ner_flair_final_studio.csv'.")

#### CSV zu EXCEL ändern

In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
csv_file_path = 'sampled_clusters_100dist_with_ner_flair_final_studio.csv'  # Replace with your CSV file path
df = pd.read_csv(csv_file_path)

# Save the DataFrame to an Excel file
excel_file_path = 'sampled_clusters_100dist_with_ner_final_flair_studio.xlsx'  # Replace with the desired output file path
df.to_excel(excel_file_path, index=False)

print(f"Excel file saved as {excel_file_path}")

### SciBert

In [ ]:
from transformers import BertTokenizer, BertForTokenClassification
from transformers import pipeline
import pandas as pd

# Load the CSV file into a DataFrame (replace with the actual path to your CSV file)
df = pd.read_csv('sampled_clusters_100dist_final_studio.csv')  # Replace with your actual file path

# Load the SciBERT model and tokenizer for NER
model_name = "allenai/scibert_scivocab_cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForTokenClassification.from_pretrained(model_name)

# Create a NER pipeline
nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer)

# Function to split text into chunks of 512 tokens and apply NER
def split_and_apply_ner(prompt):
    # Tokenize the prompt and ensure the total length does not exceed 512 tokens (special tokens included)
    tokens = tokenizer.encode(prompt, add_special_tokens=True)  # Add special tokens like [CLS] and [SEP]
    
    # Check if the length exceeds 512 and truncate, leaving room for special tokens
    if len(tokens) > 512:
        tokens = tokens[:510]  # Keep space for [CLS] and [SEP]
    
    # Decode tokens back to a string
    chunk_prompt = tokenizer.decode(tokens, skip_special_tokens=False)
    
    # Apply NER to the chunk
    ner_results = nlp_ner(chunk_prompt)
    
    # Extracting entity labels (only the entity words, not the entire result object)
    entities = [entity['word'] for entity in ner_results]
    
    # Return all entities found in the chunk
    return ", ".join(entities)

# Apply NER to 'prompt1' and 'prompt2', and store results in new columns 'ner1' and 'ner2'
print("Applying NER to Prompt 1...")
df['ner1'] = df['prompt1'].apply(split_and_apply_ner)
print("Applying NER to Prompt 2...")
df['ner2'] = df['prompt2'].apply(split_and_apply_ner)

# Save the modified DataFrame to a new CSV file (or overwrite the original file)
df.to_csv('sampled_clusters_100dist_with_scibert_ner_final_studio.csv', index=False)

# Output a message confirming the update
print("NER applied using SciBERT and saved to 'sampled_clusters_100dist_with_scibert_ner_final_studio.csv'.")

#### CSV zu EXCEL ändern

In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
csv_file_path = 'sampled_clusters_100dist_with_scibert_ner_final_studio.csv'  # Replace with your CSV file path
df = pd.read_csv(csv_file_path)

# Save the DataFrame to an Excel file
excel_file_path = 'sampled_clusters_100dist_with_scibert_ner_final_studio.xlsx'  # Replace with the desired output file path
df.to_excel(excel_file_path, index=False)

print(f"Excel file saved as {excel_file_path}")